# ⚡ Codex CLI — Guide Pratique pour Débutants

> **Basé sur** : [codex-cli-best-practice](https://github.com/shanraisshan/codex-cli-best-practice) par Shayan Raisshan  
> **Version** : Codex CLI v0.125.0 · Mai 2026  
> **Éditeur** : OpenAI

---

Codex CLI est l'**agent de développement en ligne de commande d'OpenAI** — l'équivalent de Claude Code chez Anthropic.  
Ce notebook couvre tous les concepts essentiels pour bien démarrer et comprendre ses particularités.

## Codex CLI vs Claude Code — Vue d'ensemble

| Aspect | Codex CLI (OpenAI) | Claude Code (Anthropic) |
|--------|-------------------|------------------------|
| **CLI** | `codex` | `claude` |
| **Mémoire projet** | `AGENTS.md` | `CLAUDE.md` |
| **Configuration** | `config.toml` (TOML) | `settings.json` (JSON) |
| **Agents custom** | `.codex/agents/*.toml` | `.claude/agents/*.md` |
| **Skills** | `.agents/skills/*/SKILL.md` | `.claude/skills/*/SKILL.md` |
| **Commandes** | `/plan`, `/fork`, `/review`... | `/plan`, `/compact`, `/rewind`... |
| **Mémoire cross-session** | ✅ `[features] memories = true` | ✅ Auto-memory |
| **Sandbox** | `read-only`, `workspace-write`, `danger-full-access` | `/sandbox` |
| **Profiles** | `conservative`, `development`, `trusted`, `ci` | N/A |
| **Construit en** | Rust (90%) | TypeScript |

## Plan du notebook

| # | Section | Concept clé |
|---|---------|------------|
| 1 | Installation & premiers pas | `codex`, diagnostic |
| 2 | AGENTS.md — la mémoire projet | Contexte persistant, chargement hiérarchique |
| 3 | Configuration TOML | `config.toml`, profils, sandbox, approval policy |
| 4 | Les Skills | `.agents/skills/`, auto-discovery |
| 5 | Les Sous-agents | Agents TOML, parallélisme, CSV batch |
| 6 | Les Memories | Cross-session learning |
| 7 | Les Hooks | Événements, automatisation |
| 8 | Les MCP Servers | Outils externes |
| 9 | Workflows de développement | Research → Plan → Execute → Review |
| 10 | Tips & Tricks (50 conseils) | Bonnes pratiques de l'équipe OpenAI |
| 11 | Architecture Agent → Skill | Orchestration Codex |
| 12 | Claude Code + Codex — Cross-model | Combiner les deux outils |

---
## Section 1 — Installation & Premiers Pas

### Installation

```bash
# Prérequis : Node.js 22+
npm install -g @openai/codex

# Ou via npx (sans installation globale)
npx @openai/codex

# Vérifier la version
codex --version

# Lancer dans un projet
cd mon-projet/
codex
```

### Authentification

```bash
# Via variable d'environnement (recommandé)
export OPENAI_API_KEY="sk-..."

# Ou dans config.toml
# ~/.codex/config.toml
# api_key = "$OPENAI_API_KEY"
```

### Modes de lancement

```bash
# Mode interactif (TUI)
codex

# Mode non-interactif (CI/scripts)
codex exec "résume les changements sur cette branche"

# Avec profil spécifique
codex --profile development

# Avec modèle spécifique
codex --model o3

# Fork d'une session existante
codex fork
```

### Commandes slash essentielles dans le TUI

| Commande | Rôle |
|----------|------|
| `/plan` | Entrer en mode planification (read-only) |
| `/fork` | Bifurquer la session courante pour explorer |
| `/resume` | Reprendre une session précédente |
| `/review` | Review du diff courant |
| `/fast` | Activer le mode rapide (1.5x, 2x crédits) |
| `/mcp` | Gérer les connexions MCP |
| `/agent` | Naviguer entre les threads d'agents actifs |
| `/memories` | Gérer les mémoires cross-session |
| `/permissions` | Voir/modifier les permissions |
| `/status` | Statut de la session courante |

> 💡 **Tip** : Mettez à jour Codex CLI chaque jour — `npm update -g @openai/codex`. Lisez le [changelog](https://github.com/openai/codex/releases) chaque matin.

### 📁 Où se trouvent les dossiers `.codex` et `.agents` ?

Codex CLI utilise **deux dossiers** distincts selon la portée :

#### 1. Dossier global utilisateur
```
Windows  : C:\Users\<votre_nom>\.codex\
Mac/Linux: ~/.codex/
```
Contient votre configuration personnelle partagée entre tous vos projets :
```
~/.codex/
├── config.toml          ← configuration globale personnelle
├── memories/            ← mémoires cross-session (si activées)
├── memories_extensions/ ← extensions de mémoire (plugins)
└── sessions/            ← historique des sessions
```

#### 2. Dossier projet (dans votre dépôt)
```
mon-projet/
├── .codex/              ← config Codex du projet (versionné git)
│   ├── config.toml      ← configuration de l'équipe + profils
│   ├── hooks.json       ← hooks événements
│   └── agents/          ← agents TOML personnalisés
│       ├── reviewer.toml
│       └── explorer.toml
│
└── .agents/             ← skills partagés (convention cross-tool)
    └── skills/
        └── mon-skill/
            └── SKILL.md
```

> 💡 **Pourquoi `.agents/` séparé de `.codex/` ?**  
> Le dossier `.agents/skills/` est une **convention cross-tool** — les skills sont découverts aussi bien par Codex CLI que par d'autres outils compatibles. Cela permet de réutiliser vos skills entre projets et outils.

#### Hiérarchie de priorité pour config.toml

```
Flags CLI (-c key=value)           ← priorité maximale (session uniquement)
.codex/config.toml                 ← projet (versionné git, équipe)
~/.codex/config.toml               ← personnel global
```

#### Chemin de découverte des skills

```
1. .agents/skills/           ← projet courant (priorité)
2. ~/.agents/skills/         ← global personnel  
3. /etc/codex/skills/        ← système (Linux/macOS)
```

> ⚠️ **À ajouter dans `.gitignore`** : `AGENTS.override.md` (vos surcharges personnelles de AGENTS.md)

---
## Section 2 — AGENTS.md : La Mémoire Projet

### Concept

`AGENTS.md` est l'équivalent Codex du `CLAUDE.md` de Claude Code.  
C'est le fichier de contexte persistant chargé **automatiquement** à chaque session.

### Différences clés avec CLAUDE.md

| Aspect | AGENTS.md (Codex) | CLAUDE.md (Claude Code) |
|--------|-------------------|------------------------|
| **Limite** | 32 KiB (~150 lignes recommandées) | ~200 lignes recommandées |
| **Override personnel** | `AGENTS.override.md` (non commité) | `CLAUDE.local.md` (non commité) |
| **Chargement** | Hiérarchique : cwd → git root | Ancêtres + lazy pour descendants |
| **Encodage** | UTF-8, byte-based cap | Token-based |

### Chargement hiérarchique

```
/mon-repo/
├── AGENTS.md          ← Chargé depuis la racine git
├── frontend/
│   ├── AGENTS.md      ← Chargé si cwd = frontend/
│   └── src/
└── backend/
    └── AGENTS.md      ← Chargé si cwd = backend/
```

Codex remonte du `cwd` jusqu'à la racine git et charge **tous les AGENTS.md** trouvés.  
Limite totale : **32 KiB** — après, le surplus est tronqué.

### Template AGENTS.md recommandé

```markdown
# Mon Projet

## Vue d'ensemble
API REST e-commerce. Python 3.11, FastAPI, PostgreSQL, Redis.

## Commandes essentielles
- Tests : `pytest tests/ -v --tb=short`
- Serveur dev : `uvicorn main:app --reload --port 8000`
- Lint : `ruff check . && mypy src/`
- Migration DB : `alembic upgrade head`

## Architecture
```
src/
├── api/        → Routes FastAPI
├── models/     → SQLAlchemy models  
├── services/   → Business logic
└── tests/      → pytest
```

## Conventions
- Branches : feature/xxx, fix/xxx, chore/xxx
- Commits : Conventional Commits (feat:, fix:, docs:, chore:)
- PR : squash merge, < 200 lignes diff
- Pas de print() → utiliser structlog

## Règles importantes
- Ne JAMAIS modifier les fichiers de migration existants
- Toujours ajouter des tests pour les nouvelles routes
- Les secrets via variables d'environnement uniquement
```

### AGENTS.override.md — surcharges personnelles

Créez ce fichier à la racine du projet (**non commité**) pour vos préférences :

```markdown
# AGENTS.override.md  (ajoutez à .gitignore)

## Préférences personnelles
- Répondre toujours en français
- Utiliser des f-strings plutôt que .format()
- Préférer les type hints explicites
```

> 💡 **Règle d'or** : N'importe quel développeur doit pouvoir lancer Codex, taper "run the tests" et que ça fonctionne du premier coup. Si non, votre AGENTS.md manque les commandes essentielles.

In [ ]:
# Exercice 1 : Comparer AGENTS.md vs CLAUDE.md et générer les deux
def generer_agents_md(
    nom_projet: str, stack: str,
    cmd_tests: str, cmd_serveur: str,
    conventions: list[str], regles: list[str]
) -> str:
    conv_str = "\n".join(f"- {c}" for c in conventions)
    reg_str  = "\n".join(f"- {r}" for r in regles)
    return f"""# {nom_projet}

## Vue d'ensemble
{stack}

## Commandes essentielles
- Tests   : `{cmd_tests}`
- Serveur : `{cmd_serveur}`

## Conventions
{conv_str}

## Règles importantes
{reg_str}
"""

def generer_claude_md(
    nom_projet: str, stack: str,
    cmd_tests: str, cmd_serveur: str,
    conventions: list[str], regles: list[str]
) -> str:
    conv_str = "\n".join(f"- {c}" for c in conventions)
    reg_str  = "\n".join(f"- {r}" for r in regles)
    return f"""# {nom_projet}

## Stack technique
{stack}

## Commandes importantes
- Lancer les tests : `{cmd_tests}`
- Démarrer l'app   : `{cmd_serveur}`

## Conventions de code
{conv_str}

## Instructions pour Claude
{reg_str}

<important if="modifying database">
Ne jamais modifier les migrations existantes.
</important>
"""

# ─── Paramètres communs ───────────────────────────────────────────
PROJET       = "API E-commerce"
STACK        = "Python 3.11, FastAPI, PostgreSQL, Redis"
CMD_TESTS    = "pytest tests/ -v --tb=short"
CMD_SERVEUR  = "uvicorn main:app --reload --port 8000"
CONVENTIONS  = [
    "Branches : feature/xxx, fix/xxx",
    "Commits  : Conventional Commits",
    "PR       : squash merge, < 200 lignes",
]
REGLES = [
    "Ne jamais commiter les fichiers .env",
    "Toujours ajouter des tests pour les nouvelles routes",
    "Secrets via variables d'environnement uniquement",
]

agents_md = generer_agents_md(PROJET, STACK, CMD_TESTS, CMD_SERVEUR, CONVENTIONS, REGLES)
claude_md = generer_claude_md(PROJET, STACK, CMD_TESTS, CMD_SERVEUR, CONVENTIONS, REGLES)

print("=" * 60)
print("📄 AGENTS.md  (Codex CLI)")
print("=" * 60)
print(agents_md)
print(f"Taille : {len(agents_md.encode())} octets  (limite Codex : 32 768 octets)\n")

print("=" * 60)
print("📄 CLAUDE.md  (Claude Code)")
print("=" * 60)
print(claude_md)
print(f"Lignes : {len(claude_md.splitlines())}  (recommandé Claude : < 200 lignes)")

---
## Section 3 — Configuration TOML (`config.toml`)

### La grande différence avec Claude Code

Là où Claude Code utilise du **JSON** (`settings.json`), Codex utilise du **TOML** (`config.toml`).  
Le TOML est plus lisible pour les humains, supporte les commentaires, et les tables imbriquées.

### Hiérarchie des configurations

```
Priority 1 : Flags CLI (-c key=value)          ← session uniquement
Priority 2 : .codex/config.toml                ← projet (versionné)
Priority 3 : ~/.codex/config.toml              ← global personnel
```

### config.toml minimal

```toml
# .codex/config.toml
model           = "o4-mini"
sandbox_mode    = "workspace-write"
approval_policy = "on-request"
```

### 🎛️ Les Profils — fonctionnalité unique à Codex

Les profils permettent de changer le comportement complet de Codex d'un flag.  
C'est la fonctionnalité qui n'a pas d'équivalent direct dans Claude Code.

```toml
# Profil par défaut
profile = "development"

[profiles.conservative]
# Revues, audits — lecture seule, demande pour tout
model           = "o4-mini"
sandbox_mode    = "read-only"
approval_policy = "untrusted"

[profiles.development]
# Dev quotidien — peut écrire, demande si incertain
model           = "o4-mini"
sandbox_mode    = "workspace-write"
approval_policy = "on-request"

[profiles.trusted]
# Automation complète — accès total, ne demande jamais
model           = "o3"
sandbox_mode    = "danger-full-access"
approval_policy = "never"

[profiles.ci]
# Pipeline CI — lecture seule, non-interactif
model           = "o4-mini"
sandbox_mode    = "read-only"
approval_policy = "never"

[profiles.review]
# Review de PR — lecture + on-request
model           = "o3"
sandbox_mode    = "read-only"
approval_policy = "on-request"
```

**Utilisation :**
```bash
codex --profile conservative    # audit de sécurité
codex --profile development     # dev quotidien
codex --profile trusted         # automation longue nuit
codex --profile ci              # pipeline GitHub Actions
```

### 🏖️ Sandbox Modes

| Mode | Fichiers | Réseau | Utilisation |
|------|---------|--------|------------|
| `read-only` | Lecture seule | Bloqué | Reviews, audits |
| `workspace-write` | Lecture/écriture dans workspace | Bloqué | Dev local |
| `danger-full-access` | Accès illimité | Autorisé | Automation de confiance |

### ✅ Approval Policies

| Policy | Comportement | Pour quand |
|--------|-------------|-----------|
| `untrusted` | Demande pour tout sauf lecture | Nouveaux repos, repos inconnus |
| `on-request` | Le modèle décide quand demander | Dev quotidien (recommandé pour débuter) |
| `never` | N'interrompt jamais | CI, automation non-interactive |

> ⚠️ **Anti-patterns** :
> - Ne jamais utiliser `danger-full-access` + `never` ensemble sans isolation réelle
> - Ne pas hardcoder les secrets — utiliser `$ENV_VAR` dans le TOML

In [ ]:
import json

# Exercice 2 : Générateur de config.toml avec profils

def generer_config_toml(
    model_default: str = "o4-mini",
    sandbox_default: str = "workspace-write",
    approval_default: str = "on-request",
    profil_par_defaut: str = "development",
    avec_memories: bool = True,
    mcp_servers: dict = None,
) -> str:

    memories_section = ""
    if avec_memories:
        memories_section = """
[features]
memories = true

[memories]
use_memories      = true
generate_memories = true
no_memories_if_mcp_or_web_search = false
"""

    mcp_section = ""
    if mcp_servers:
        for nom, config in mcp_servers.items():
            mcp_section += f"\n[mcp_servers.{nom}]\n"
            for k, v in config.items():
                if isinstance(v, str):
                    mcp_section += f'{k} = "{v}"\n'
                elif isinstance(v, list):
                    args = ", ".join(f'"{a}"' for a in v)
                    mcp_section += f'{k} = [{args}]\n'

    return f"""# .codex/config.toml
# Généré automatiquement — personnalisez selon vos besoins

model           = "{model_default}"
sandbox_mode    = "{sandbox_default}"
approval_policy = "{approval_default}"
profile         = "{profil_par_defaut}"

# ──────────────── Profils ────────────────────────────────────────

[profiles.conservative]
sandbox_mode    = "read-only"
approval_policy = "untrusted"

[profiles.development]
model           = "o4-mini"
sandbox_mode    = "workspace-write"
approval_policy = "on-request"

[profiles.trusted]
model           = "o3"
sandbox_mode    = "danger-full-access"
approval_policy = "never"

[profiles.ci]
model           = "o4-mini"
sandbox_mode    = "read-only"
approval_policy = "never"

[profiles.review]
model           = "o3"
sandbox_mode    = "read-only"
approval_policy = "on-request"

# ──────────────── Agents ─────────────────────────────────────────

[agents]
max_threads               = 6
max_depth                 = 1
job_max_runtime_seconds   = 1800
{memories_section}{mcp_section}"""


config = generer_config_toml(
    avec_memories=True,
    mcp_servers={
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
        }
    }
)
print(config)

# ─── Guide de sélection de profil ────────────────────────────────
print("=" * 55)
print("🎯 Quel profil utiliser ?")
print("=" * 55)
cas = [
    ("Je commence sur un repo inconnu",           "conservative",  "codex --profile conservative"),
    ("Dev quotidien sur mon projet",               "development",   "codex --profile development"),
    ("Laisser tourner toute la nuit sans surveiller", "trusted",   "codex --profile trusted"),
    ("Pipeline GitHub Actions",                    "ci",            "codex --profile ci"),
    ("Review d'une PR complexe",                   "review",        "codex --profile review"),
]
for situation, profil, cmd in cas:
    print(f"\n  📌 {situation}")
    print(f"     Profil  : {profil}")
    print(f"     Commande: {cmd}")

---
## Section 4 — Les Skills

### Concept

Les Skills de Codex sont **identiques dans le principe** à ceux de Claude Code — des paquets d'instructions réutilisables avec frontmatter YAML.

**Différence principale** : le dossier est `.agents/skills/` (pas `.claude/skills/`).

```
.agents/
└── skills/
    └── mon-skill/
        ├── SKILL.md          ← Fichier principal (obligatoire)
        ├── references/       ← Documentation de référence
        ├── scripts/          ← Scripts réutilisables
        └── assets/           ← Ressources (images, données)
```

### Découverte des skills

Codex découvre les skills depuis plusieurs emplacements (ordre de priorité) :

```
1. ~/.agents/skills/           ← global personnel
2. .agents/skills/             ← projet courant
3. /etc/codex/skills/          ← système (Linux/macOS)
4. Plugin skills               ← depuis les plugins installés
```

### Structure d'un SKILL.md

```yaml
---
name: api-tester
description: Tests REST API endpoints using curl. Use when verifying API responses or debugging endpoints.
---

## Objectif
Tester des endpoints REST et valider les réponses.

## Utilisation
Appelle l'endpoint avec curl, vérifie le status code et la structure JSON.

## Gotchas
- Toujours suivre les redirects avec `-L`
- Utiliser `-s` pour supprimer la barre de progression dans les scripts
- Headers JSON : `-H "Content-Type: application/json"`

## Exemple
```bash
curl -s -L -X POST http://localhost:8000/api/users \
  -H "Content-Type: application/json" \
  -d '{"email": "test@example.com"}' | jq .
```
```

### Invoquer un skill

```bash
# Invocation explicite dans le TUI
$api-tester

# Ou via le menu
/skills → sélectionner api-tester

# Invocation automatique (si description match)
"Test the login endpoint"
→ Codex détecte que api-tester correspond et l'invoque
```

### Skills intégrés

| Skill | Invocation | Usage |
|-------|-----------|-------|
| `$plan` | `$plan` | Planification détaillée avant implémentation |
| `$skill-creator` | `$skill-creator` | Crée un nouveau skill scaffoldé |
| `$skill-installer` | `$skill-installer` | Installe un skill depuis une URL/chemin |

### Bonnes pratiques (équipe OpenAI)

1. **Description = déclencheur** — écrivez "quand invoquer" pas "ce que ça fait"
2. **Section Gotchas** = contenu le plus précieux du skill
3. **Ne pas railroader** — objectifs + contraintes, pas de pas-à-pas rigide
4. **Progressive disclosure** — core dans SKILL.md, détails dans `references/`
5. **Ne pas énoncer l'évident** — concentrez-vous sur ce qui sort Codex de son comportement par défaut

---
## Section 5 — Les Sous-agents (Subagents)

### Concept

Codex peut spawner des **sous-agents spécialisés en parallèle** — sa fonctionnalité multi-agent est particulièrement puissante pour les reviews de PR et le débogage.

**Différence clé vs Claude Code** :
- Claude Code : agents définis en `.md` (Markdown)
- Codex CLI : agents définis en `.toml` (TOML), dans `.codex/agents/`

### Agents intégrés (3)

| Agent | Rôle |
|-------|------|
| `default` | Agent général polyvalent |
| `worker` | Orienté exécution — implémentation, corrections |
| `explorer` | Orienté lecture — exploration de codebase |

### Structure d'un agent custom (TOML)

```toml
# .codex/agents/code-reviewer.toml

name        = "reviewer"
description = "PR reviewer focused on correctness, security, and missing tests"
model       = "o3"
model_reasoning_effort = "high"
sandbox_mode = "read-only"

developer_instructions = """
Review code like an owner.
Prioritize correctness, security, behavior regressions, and missing test coverage.
Lead with concrete findings, include reproduction steps when possible.
Avoid style-only comments unless they hide a real bug.
"""

# Noms affichés quand plusieurs instances tournent
nickname_candidates = ["Atlas", "Delta", "Echo"]
```

### Champs TOML des agents

| Champ | Type | Description |
|-------|------|-------------|
| `name` | string | **Requis** — identifiant |
| `description` | string | **Requis** — quand Codex l'utilise |
| `developer_instructions` | string | **Requis** — instructions de comportement |
| `model` | string | Modèle dédié (hérite si omis) |
| `model_reasoning_effort` | string | `low`, `medium`, `high` |
| `sandbox_mode` | string | Override du sandbox pour cet agent |
| `nickname_candidates` | list | Noms lisibles pour plusieurs instances |
| `mcp_servers` | table | MCP servers pour cet agent |
| `skills.config` | list | Skills actifs pour cet agent |

### Le pattern de PR review parallèle

```
Prompt :
"Review this branch vs main. Spawn one agent per concern,
 wait for all, and summarize."

Agents lancés en parallèle :
├── reviewer → correctness & security (model: o3, high effort)
├── pr-explorer → map code paths affected (model: gpt-5.3, read-only)
└── docs-researcher → verify framework APIs (model: gpt-5.3 + MCP docs)
```

### Paramètres globaux des agents (config.toml)

```toml
[agents]
max_threads             = 6     # Agents parallèles max
max_depth               = 1     # Profondeur d'imbrication (0 = root)
job_max_runtime_seconds = 1800  # Timeout par worker (30 min)
```

### CSV Batch Processing — feature avancée

Pour des tâches massives répétitives :

```
Prompt :
"Create /tmp/components.csv with path,owner columns.
 Then call spawn_agents_on_csv to review each component.
 Output: /tmp/components-review.csv"
```

Codex lit le CSV, spawne **un worker par ligne**, attend tous les résultats, exporte un CSV de sortie.

### Navigation entre agents dans le TUI

```
/agent          → liste tous les threads actifs
→ sélectionner  → switcher vers ce thread
press 'o'       → ouvrir le thread d'une approval request
```

> 💡 **Tip** : Utilisez `nickname_candidates` pour distinguer visuellement plusieurs instances du même agent (`Atlas reviewing security`, `Delta reviewing tests`...).

In [ ]:
# Exercice 3 : Générateur d'agents TOML pour Codex

def generer_agent_toml(
    name: str,
    description: str,
    instructions: str,
    model: str = "o4-mini",
    reasoning_effort: str = "medium",
    sandbox: str = "read-only",
    nicknames: list[str] = None,
    mcp_server: dict = None,
) -> str:
    nicks = ""
    if nicknames:
        nicks_str = ", ".join(f'"{n}"' for n in nicknames)
        nicks = f'\nnickname_candidates = [{nicks_str}]'

    mcp_section = ""
    if mcp_server:
        nom = mcp_server["name"]
        url = mcp_server.get("url", "")
        mcp_section = f'\n[mcp_servers.{nom}]\nurl = "{url}"\n'

    return f"""# .codex/agents/{name}.toml

name                   = "{name}"
description            = "{description}"
model                  = "{model}"
model_reasoning_effort = "{reasoning_effort}"
sandbox_mode           = "{sandbox}"{nicks}

developer_instructions = \"\"\"
{instructions.strip()}
\"\"\"
{mcp_section}"""


agents_equipe = [
    {
        "name": "pr-explorer",
        "description": "Read-only codebase explorer. Use to map affected code paths before reviewing changes.",
        "model": "o4-mini",
        "reasoning_effort": "medium",
        "sandbox": "read-only",
        "nicknames": ["Scout", "Ranger"],
        "instructions": """Map the code that the current changes touch.
Trace real execution paths, cite files and line numbers.
Avoid proposing fixes — only gather evidence.""",
    },
    {
        "name": "reviewer",
        "description": "PR code reviewer focused on bugs, security, and test coverage.",
        "model": "o3",
        "reasoning_effort": "high",
        "sandbox": "read-only",
        "nicknames": ["Atlas", "Delta", "Echo"],
        "instructions": """Review code like a senior engineer.
Prioritize: correctness, security vulnerabilities, behavior regressions, missing tests.
Report: 🔴 Critical / 🟡 Important / 🟢 Suggestion.
Include reproduction steps for bugs.""",
    },
    {
        "name": "ui-fixer",
        "description": "Frontend bug fixer. Use after issue is reproduced and code paths are mapped.",
        "model": "o4-mini",
        "reasoning_effort": "medium",
        "sandbox": "workspace-write",
        "instructions": """Apply the smallest defensible fix to the identified issue.
Keep unrelated files untouched.
Validate only the behavior you changed.
Do not introduce new dependencies without explicit approval.""",
    },
]

for agent_config in agents_equipe:
    toml = generer_agent_toml(**agent_config)
    print(f"{'='*60}")
    print(f"📄 .codex/agents/{agent_config['name']}.toml")
    print('='*60)
    print(toml)
    print()

print("🚀 Prompt d'orchestration :")
print("""
  Review this branch vs main.
  1. Have pr-explorer map all affected code paths
  2. Have reviewer find real bugs and security issues
  3. If UI issues found, have ui-fixer apply the smallest fix
  Spawn in parallel, wait for all results, then summarize.
""")

---
## Section 6 — Les Memories (Mémoire Cross-session)

### Concept

Les **Memories** de Codex sont une fonctionnalité **unique** — absent dans Claude Code par défaut.  
Elles apprennent de vos sessions et injectent des notes pertinentes dans les futures.

```
Session 1 : vous expliquez votre style de code à Codex
     ↓ (consolidation entre sessions, en arrière-plan)
Session 2 : Codex se souvient de votre style et l'applique automatiquement
```

### Activer les memories

```toml
# config.toml
[features]
memories = true

[memories]
use_memories      = true    # injecter dans les futures sessions
generate_memories = true    # enregistrer les sessions actuelles
no_memories_if_mcp_or_web_search = false   # garde-fou sécurité
```

### Contrôles TUI

```
/memories → ouvre le panneau
    ├── Use memories    [ON/OFF]  ← injecter dans ce thread
    ├── Generate memories [ON/OFF] ← enregistrer ce thread  
    └── Reset all memories        ← effacer toutes les mémoires
```

### Où sont stockées les memories

```
~/.codex/memories/              ← mémoires globales (per-user, pas per-project)
├── raw_memories/               ← brouillons extraits
└── summaries/                  ← résumés consolidés

~/.codex/memories_extensions/   ← extensions de mémoire
└── <plugin>/
    ├── instructions.md         ← toujours chargé
    └── resources/              ← par session, nettoyé après 7j
```

⚠️ **Important** : Les memories sont **par utilisateur**, pas par projet.  
Pour du contexte projet, utilisez `AGENTS.md`.

### Pipeline de consolidation

```
Session termine
     ↓
Phase 1 (extract_model: gpt-5.4-mini)
  → Extrait des "raw memories" de la session

Phase 2 (consolidation_model: gpt-5.4)
  → Fusionne + résume les raw memories
  → Nettoie les resources > 7j

Prochaine session
  → use_memories injecte les summaries pertinents
```

### Bonnes pratiques

| Faire ✅ | Éviter ❌ |
|----------|---------|
| Activer une fois, oublier | Éditer manuellement les fichiers memories |
| `no_memories_if_mcp_or_web_search = true` pour les threads sensibles | Stocker des secrets attendant un nettoyage auto |
| Reset si changement de domaine/rôle | Utiliser memories comme base de connaissances structurée |
| CI : `[features] memories = false` | Attendre des memories per-project (→ utilisez AGENTS.md) |

---
## Section 7 — Les Hooks

### Concept

Les hooks Codex sont des **scripts shell déclenchés par des événements** dans la boucle agentique.  
Ils permettent le logging, la validation, la sécurité, l'auto-formatting, etc.

⚠️ **Feature flag requis** :
```toml
[features]
codex_hooks = true
```

### Les 5 événements de hooks Codex

| Événement | Déclenchement |
|-----------|--------------|
| `SessionStart` | Démarrage d'une session (ou reprise) |
| `PreToolCall` | Avant qu'un outil (bash, file, etc.) soit exécuté |
| `PostToolCall` | Après qu'un outil soit exécuté |
| `TurnEnd` | Fin d'un tour de l'agent |
| `SessionEnd` | Fin de session |

*Comparaison : Claude Code a 27 événements de hooks, Codex en a 5.*

### Structure `.codex/hooks.json`

```json
{
  "hooks": [
    {
      "event": "PreToolCall",
      "command": "python .codex/hooks/pre-tool.py",
      "timeout_seconds": 10
    },
    {
      "event": "PostToolCall",
      "command": "python .codex/hooks/post-tool.py",
      "timeout_seconds": 10
    },
    {
      "event": "SessionStart",
      "command": "python .codex/hooks/session-start.py",
      "timeout_seconds": 5
    }
  ]
}
```

### Cas d'usage typiques

```python
# .codex/hooks/post-tool.py — Auto-format après modification de fichier
import json, sys, subprocess

event = json.load(sys.stdin)
if event.get("tool") == "write_file":
    path = event.get("path", "")
    if path.endswith(".py"):
        subprocess.run(["ruff", "format", path])
    elif path.endswith((".ts", ".js")):
        subprocess.run(["prettier", "--write", path])
```

```python
# .codex/hooks/session-start.py — Notification sonore au démarrage
import json, sys, subprocess, platform

event = json.load(sys.stdin)
source = event.get("source", "startup")  # startup | resume | clear

# Skip sur /clear (contexte réinitialisé, pas de son lourd)
if source == "clear":
    sys.exit(0)

# Notification selon OS
if platform.system() == "Darwin":
    subprocess.run(["afplay", "/System/Library/Sounds/Hero.aiff"])
```

### Tip avancé : brancher sur `source` dans SessionStart

```python
source = event.get("source")  # "startup" | "resume" | "clear"

if source == "startup":
    # Chargement complet du contexte — OK pour la lourdeur
    load_project_context()
elif source == "resume":
    # Session reprise — contexte existe déjà
    light_refresh()
elif source == "clear":
    # /clear tapé — ne pas recharger, rester snappy
    pass
```

> 💡 **Tip** : Utilisez les hooks pour l'auto-formatting — Codex génère du code bien formaté à 90%, le hook finit les 10% restants pour éviter les échecs CI.

---
## Section 8 — MCP Servers

### Concept

Les MCP (Model Context Protocol) servers connectent Codex à des **outils externes** :  
bases de données, GitHub, navigateur, documentation, APIs...

Codex peut aussi **servir comme MCP server** pour être appelé par d'autres outils.

### Déclarer un MCP server (STDIO)

```toml
# config.toml

[mcp_servers.github]
command = "npx"
args    = ["-y", "@modelcontextprotocol/server-github"]
env     = { GITHUB_TOKEN = "$GITHUB_TOKEN" }

[mcp_servers.playwright]
command = "npx"
args    = ["-y", "@playwright/mcp@latest"]
```

### Déclarer un MCP server (HTTP)

```toml
[mcp_servers.openai-docs]
url                 = "https://developers.openai.com/mcp"
startup_timeout_sec = 20
```

### MCP servers utiles

| MCP Server | Usage | Installation |
|-----------|-------|-------------|
| `@modelcontextprotocol/server-github` | GitHub Issues, PRs, repos | `npx -y @modelcontextprotocol/server-github` |
| `@playwright/mcp` | Browser automation | `npx -y @playwright/mcp@latest` |
| `chrome-devtools-mcp` | Console logs Chrome | Via DevTools |
| `openai-docs` | Docs officielles OpenAI | URL MCP |

### Codex comme serveur MCP

```bash
# Exposer Codex comme outil pour d'autres agents
codex mcp-server

# Outils exposés :
# codex()        → exécute une session Codex
# codex-reply()  → reply dans une session existante
```

### Gérer les MCP servers

```bash
# Ajouter un serveur
codex mcp add github npx -y @modelcontextprotocol/server-github

# Lister les serveurs
codex mcp list

# Authentification OAuth
codex mcp login github

# Supprimer
codex mcp remove github
```

### MCP Apps (v0.119.0+)

```toml
# Support pour les apps MCP avancées
[mcp_servers.my-app]
url = "http://localhost:3000/mcp"
supports_parallel_tool_calls = true  # v0.121.0+
```

Les MCP Apps peuvent faire des **resource reads**, **elicitations** (demander des fichiers à l'utilisateur), et **file-parameter uploads**.

---
## Section 9 — Workflows de Développement

### Le pattern universel (identique à Claude Code)

```
Research → Plan → Execute → Review → Ship
```

### Workflow Codex étape par étape

#### Étape 1 : Spécifier avec détail

```
"Crée un endpoint POST /api/orders qui :
 - Valide le panier (stock, prix)
 - Crée la commande en DB
 - Déclenche l'email de confirmation
 - Réponds aux edge cases (stock insuffisant, paiement échoué)
Écris des tests d'intégration."
```

#### Étape 2 : Planifier avec `/plan`

```
/plan
→ Codex passe en mode read-only
→ Explore votre codebase existant
→ Propose une architecture détaillée avec les fichiers à créer
→ Vous validez avant tout changement
```

#### Étape 3 : Exécuter avec sous-agents parallèles

```
"Implémente le plan validé.
 Spawn separate agents for:
 - The order service implementation
 - The integration tests
 - The API documentation
 Wait for all agents and report back."
```

#### Étape 4 : Review multi-agents

```
"Review this implementation.
 Spawn agents for:
 - Security (injection, auth bypass)
 - Correctness (edge cases, error handling)
 - Test coverage (missing scenarios)
 Use the reviewer agent for each."
```

#### Étape 5 : Ship

```bash
# Via codex exec pour une review finale non-interactive
codex --profile review exec "Is this branch ready to merge? List any blockers."
```

### Modèles Codex : quand utiliser quoi

| Modèle | Force | Usage |
|--------|-------|-------|
| `gpt-5.3-codex-spark` | Ultra-rapide | Exploration, recherche, itérations UI |
| `o4-mini` | Équilibré | Dev quotidien, corrections, refactoring |
| `o3` | Maximum de raisonnement | Architecture, débogage complexe, review critique |

### `/fork` — explorer sans perdre le fil

```
Session principale → /fork
                          ├── Branche A : essaie l'implémentation X
                          └── Branche B : essaie l'implémentation Y
                                ↓
                     /resume → choisit la meilleure branche
```

### `codex exec` — mode non-interactif pour CI

```yaml
# .github/workflows/codex-review.yml
- name: Codex security review
  run: |
    codex --profile ci exec "Review the diff for security vulnerabilities. 
    Exit 1 if critical issues found."
  env:
    OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
```

---
## Section 10 — Tips & Tricks (50 conseils)

### 🎯 Prompting

| Conseil |
|---------|
| "Fix" suffit — collez le bug, dites "fix", ne micromanagez pas |
| Après un fix médiocre : "Knowing everything you know now, scrap this and implement the elegant solution" |
| "Prove to me this works" — challengez Codex comme un reviewer exigeant |

### 📋 Planning

| Conseil |
|---------|
| Toujours `/plan` avant les tâches complexes — Codex peut aussi planifier automatiquement |
| Plans phasés avec tests à chaque phase (unit, automation, integration) |
| Spécifications détaillées = meilleure sortie — réduire l'ambiguïté avant de déléguer |
| Spinup un second Codex (ou Claude Code) pour critiquer votre plan |

### 📄 AGENTS.md

| Conseil |
|---------|
| Sous 150 lignes (hard limit : 32 KiB) |
| `AGENTS.override.md` pour les préférences personnelles (ne pas commiter) |
| N'importe qui doit pouvoir lancer Codex et taper "run the tests" sans setup |
| `config.toml` pour les comportements déterministes — pas AGENTS.md |
| Finir les migrations — un codebase à moitié migré confuse le modèle |

### 🤖 Agents

| Conseil |
|---------|
| Agents spécifiques + skills > agents généraux |
| "Spawn subagents" = plus de compute sur le problème |
| Contextes séparés = meilleurs résultats — un agent crée le bug, un autre le trouve |
| Nommez vos agents avec `nickname_candidates` pour les distinguer visuellement |

### 🛠️ Skills

| Conseil |
|---------|
| `description` = déclencheur, pas résumé |
| Section **Gotchas** = contenu le plus précieux |
| Ne pas railroader — objectifs + contraintes, pas de pas-à-pas |
| Progressive disclosure : core dans SKILL.md, détails dans `references/` |
| Utilisez `$skill-creator` pour scaffolder vos nouveaux skills |

### 🧠 Memories

| Conseil |
|---------|
| Activez une fois, oubliez — la consolidation tourne en arrière-plan |
| `no_memories_if_mcp_or_web_search = true` pour les threads sensibles |
| Reset si changement de domaine/rôle — les vieilles mémoires nuisent |

### ⚙️ Workflows

| Conseil |
|---------|
| Codex vanilla bat les workflows complexes pour les petites tâches |
| Profil `on-request` pour débuter — n'escaladez vers `never` que progressivement |
| `/fork` pour explorer des alternatives sans perdre votre thread courant |
| `codex exec` pour CI/scripts non-interactifs |

### ⚡ Workflows Avancés

| Conseil |
|---------|
| Multi-agents pour le travail parallèle (fan-out) |
| `workspace-write` + `on-request` = bon défaut pour le dev local |
| Git worktrees pour le développement en parallèle |
| ASCII diagrams pour comprendre votre architecture |

### 🐛 Debugging

| Conseil |
|---------|
| Screenshots → partager avec Codex quand bloqué |
| MCP Chrome DevTools ou Playwright pour que Codex voit les console logs |
| Lancer le terminal de logs en arrière-plan pour mieux déboguer |
| Utilisez Claude Code pour la review de plan — cross-model QA |

### 🛠️ Git

| Conseil |
|---------|
| PRs petites et focalisées (une feature, une PR) |
| Squash merge pour un historique propre |
| Committer souvent — dès qu'une tâche est terminée |

---
## Section 11 — Architecture Agent → Skill

### Le pattern d'orchestration Codex

Codex suit le pattern **Agent → Skill** (pas Command → Agent → Skill comme Claude Code,  
car les custom commands `.codex/commands/` ne sont pas encore disponibles).

```
Utilisateur
    │
    │ "Fetch weather for Dubai and create SVG card"
    ▼
Codex (session principale)
    │
    │ Invoke weather-agent
    ▼
.codex/agents/weather-agent.toml
    │  (agent spécialisé météo)
    │
    ├── Appelle Open-Meteo API → température
    │
    └── Invoque skill $weather-svg-creator
            └── .agents/skills/weather-svg-creator/SKILL.md
                    → Génère orchestration-workflow/weather.svg
```

### Exemple complet — Agent météo

```toml
# .codex/agents/weather-agent.toml

name        = "weather-agent"
description = "Fetches weather data and delegates SVG rendering to the weather-svg-creator skill"
model       = "o4-mini"
sandbox_mode = "read-only"

developer_instructions = """
You are a weather data specialist.

1. Fetch current temperature for the requested city from Open-Meteo API
2. Parse the JSON response to extract temperature and windspeed
3. Invoke the $weather-svg-creator skill with the data
4. Report the final SVG path to the user

Always use metric units (Celsius) unless user requests Fahrenheit.
"""
```

```yaml
# .agents/skills/weather-svg-creator/SKILL.md
---
name: weather-svg-creator
description: Creates an SVG weather card from temperature data. Use when weather data needs visual display.
---

## Objectif
Créer une carte SVG visuelle à partir des données météo.

## Paramètres attendus
- temperature: float (Celsius ou Fahrenheit)
- windspeed: float (km/h)
- city: string

## Sortie
- Écrire dans orchestration-workflow/weather.svg
- Mettre à jour orchestration-workflow/output.md

## Gotchas
- L'heure locale dans le SVG doit utiliser le fuseau de la ville
- Convertir F→C si température en Fahrenheit
```

### Différence structurelle : Codex vs Claude Code

```
Claude Code :
Commande (.claude/commands/) 
    → Agent (.claude/agents/) 
        → Skill (.claude/skills/)

Codex CLI :
[pas de custom commands encore]
    → Agent (.codex/agents/*.toml)
        → Skill (.agents/skills/)
```

### Structure de projet complète Codex

```
mon-projet/
├── AGENTS.md                      ← Mémoire projet (< 150 lignes / 32 KiB)
├── AGENTS.override.md             ← Surcharges perso (gitignore)
├── .codex/
│   ├── config.toml                ← Config + profils + MCP
│   ├── hooks.json                 ← Hooks événements
│   └── agents/
│       ├── reviewer.toml          ← Agent review
│       ├── pr-explorer.toml       ← Agent exploration
│       └── ui-fixer.toml          ← Agent corrections UI
└── .agents/
    └── skills/
        ├── api-tester/
        │   └── SKILL.md
        └── doc-generator/
            ├── SKILL.md
            └── references/
                └── style-guide.md
```

---
## Section 12 — Claude Code + Codex : Workflow Cross-Model

### Pourquoi combiner les deux ?

Chaque outil a ses forces — les combiner donne le meilleur des deux mondes :

| Tâche | Outil recommandé | Pourquoi |
|-------|-----------------|---------|
| Planification architecture | **Claude Code** (Opus) | Raisonnement supérieur en mode plan |
| Implémentation code | **Codex** (o3) | Spécialisé coding, fast mode |
| Review sécurité | **Claude Code** `/security-review` | Très bon en audit sécurité |
| Review correctness | **Codex** (reviewer agent) | Parallélisme multi-agent natif |
| Debug frontend | **Codex** (browser-debugger + playwright MCP) | MCP apps intégrés |
| Mémoire cross-session | **Codex** | Feature native memories |
| Commandes personnalisées | **Claude Code** | `.claude/commands/` matures |

### Workflow cross-model recommandé

```
Terminal 1 (Claude Code)          Terminal 2 (Codex)
       │                                  │
  /plan → architecture                    │
  Opus raisonne                           │
       │                                  │
  Export plan.md ──────────────────────→  │
                                    Implémente le plan
                                    o3 + sous-agents
                                          │
                     ←──────── PR créée ──│
  /security-review                        │
  /ultrareview                            │
       │                                  │
  Approved ─────────────────────────────→ Merge
```

### Mécanismes d'intégration

**1. Plugin OpenAI dans Claude Code**
```bash
# .claude/commands/ contient maintenant :
/codex:review           ← review Codex depuis Claude Code
/codex:adversarial-review
/codex:rescue
```

**2. Codex comme MCP server**
```bash
codex mcp-server
# Claude Code peut appeler Codex via MCP
```

**3. Workflow manuel deux terminaux**
```bash
# Terminal 1 : Claude Code (planification)
claude
/plan feature X

# Terminal 2 : Codex (implémentation)
codex --profile development
"Implement the plan from plan.md"
```

### Ressources cross-model

- [Cross-Model Workflow Guide](https://github.com/shanraisshan/claude-code-best-practice/blob/main/development-workflows/cross-model-workflow/cross-model-workflow.md)
- [Plugin OpenAI officiel pour Claude Code](https://github.com/openai/codex-plugin-cc)
- [claude-code-router](https://github.com/musistudio/claude-code-router) — router pour utiliser différents modèles

In [ ]:
# Exercice final : Générateur de structure .codex/ complète

import json

def generer_structure_codex(
    projet: str,
    stack: str,
    cmd_tests: str,
    agents_voulus: list[str],
    avec_memories: bool = True,
    mcp_servers: list[str] = None,
) -> dict:
    """Génère l'ensemble des fichiers .codex/ et .agents/ pour démarrer."""

    fichiers = []

    # AGENTS.md
    fichiers.append({
        "chemin": "AGENTS.md",
        "contenu": f"# {projet}\n\n## Stack\n{stack}\n\n## Tests\n- `{cmd_tests}`\n\n## Conventions\n- [À compléter]"
    })

    # config.toml
    mcp_block = ""
    if mcp_servers:
        for srv in mcp_servers:
            mcp_block += f'\n[mcp_servers.{srv}]\ncommand = "npx"\nargs = ["-y", "@modelcontextprotocol/server-{srv}"]\n'

    memories_block = ""
    if avec_memories:
        memories_block = "\n[features]\nmemories = true\n\n[memories]\nuse_memories = true\ngenerate_memories = true\n"

    fichiers.append({
        "chemin": ".codex/config.toml",
        "contenu": f"""model           = "o4-mini"
sandbox_mode    = "workspace-write"
approval_policy = "on-request"
profile         = "development"

[agents]
max_threads = 6
max_depth   = 1

[profiles.conservative]
sandbox_mode    = "read-only"
approval_policy = "untrusted"

[profiles.development]
sandbox_mode    = "workspace-write"
approval_policy = "on-request"

[profiles.trusted]
model           = "o3"
sandbox_mode    = "danger-full-access"
approval_policy = "never"

[profiles.ci]
sandbox_mode    = "read-only"
approval_policy = "never"
{memories_block}{mcp_block}"""
    })

    # hooks.json
    fichiers.append({
        "chemin": ".codex/hooks.json",
        "contenu": json.dumps({
            "hooks": [
                {"event": "PostToolCall", "command": "python .codex/hooks/post-tool.py", "timeout_seconds": 10},
                {"event": "SessionStart", "command": "python .codex/hooks/session-start.py", "timeout_seconds": 5}
            ]
        }, indent=2)
    })

    # Agents
    agent_templates = {
        "reviewer": "PR reviewer focused on correctness, security, and missing tests",
        "pr-explorer": "Read-only codebase explorer for mapping affected code paths",
        "ui-fixer": "Implementation-focused agent for small targeted fixes",
    }
    for nom in agents_voulus:
        desc = agent_templates.get(nom, f"Agent spécialisé : {nom}")
        fichiers.append({
            "chemin": f".codex/agents/{nom}.toml",
            "contenu": f'name        = "{nom}"\ndescription = "{desc}"\nmodel       = "o4-mini"\nsandbox_mode = "read-only"\n\ndeveloper_instructions = """\n[À compléter : instructions de comportement]\n"""'
        })

    return {"projet": projet, "fichiers": fichiers}


# ─── Générer pour un projet exemple ──────────────────────────────
struct = generer_structure_codex(
    projet="Mon App React + FastAPI",
    stack="Python 3.11 + FastAPI / TypeScript + React / PostgreSQL",
    cmd_tests="pytest tests/ -v && npm test",
    agents_voulus=["reviewer", "pr-explorer"],
    avec_memories=True,
    mcp_servers=["github"],
)

print(f"=== Structure Codex CLI pour '{struct['projet']}' ===\n")
for f in struct["fichiers"]:
    print(f"  📄 {f['chemin']}")

print("\n\n=== .codex/config.toml ===")
for f in struct["fichiers"]:
    if f["chemin"] == ".codex/config.toml":
        print(f["contenu"])

print("\n✅ Copiez ces fichiers dans votre projet !")
print("💡 Commandes de démarrage :")
print("   codex --profile development")
print("   codex --profile conservative   (nouveau repo inconnu)")
print("   codex --profile ci exec 'review for blockers'  (CI)")

---
## Récapitulatif & Checklist de démarrage

### Ce que vous avez appris

| Concept | Fichier/Emplacement | Équivalent Claude Code |
|---------|---------------------|----------------------|
| **AGENTS.md** | Racine du projet | `CLAUDE.md` |
| **config.toml** | `.codex/config.toml` | `settings.json` |
| **Profils** | `[profiles.xxx]` dans config.toml | *(pas d'équivalent direct)* |
| **Agents custom** | `.codex/agents/*.toml` | `.claude/agents/*.md` |
| **Skills** | `.agents/skills/*/SKILL.md` | `.claude/skills/*/SKILL.md` |
| **Memories** | `[features] memories = true` | Auto-memory |
| **Hooks** | `.codex/hooks.json` | `.claude/settings.json` → hooks |
| **MCP** | `[mcp_servers.*]` | `[mcpServers]` |

### Checklist de démarrage Codex CLI

```
□ 1.  Installer : npm install -g @openai/codex
□ 2.  Configurer OPENAI_API_KEY dans l'environnement
□ 3.  Créer AGENTS.md (< 150 lignes, commandes de test incluses)
□ 4.  Créer .codex/config.toml avec les 5 profils
□ 5.  Démarrer avec --profile development (défaut quotidien)
□ 6.  Utiliser /plan pour les nouvelles features
□ 7.  Activer les memories une fois : [features] memories = true
□ 8.  Créer 2-3 agents pour review/exploration
□ 9.  Ajouter un MCP GitHub si vous travaillez sur des PRs
□ 10. Committer souvent + squash merge + PRs < 200 lignes
□ 11. Mettre à jour Codex CLI chaque jour
```

### Ressources

- **Repo source** : [codex-cli-best-practice](https://github.com/shanraisshan/codex-cli-best-practice)
- **Docs officielles** : [developers.openai.com/codex](https://developers.openai.com/codex/overview)
- **Plugin OpenAI pour Claude Code** : [codex-plugin-cc](https://github.com/openai/codex-plugin-cc)
- **Hooks Codex** : [codex-cli-hooks](https://github.com/shanraisshan/codex-cli-hooks)
- **Community** : [r/ChatGPT](https://www.reddit.com/r/ChatGPT/) · [r/OpenAI](https://www.reddit.com/r/OpenAI/)

### Podcasts recommandés

| Podcast | Intervenants | Pour qui |
|---------|-------------|---------|
| [How Codex team uses their coding agent](https://every.to/podcast/transcript-how-openai-s-codex-team-uses-their-coding-agent) | Tibo + Andrew | Tous niveaux |
| [The power user's guide to Codex](https://open.spotify.com/episode/6RNqTaOb5ly3zgQCGB23fE) | Embiricos | Utilisateurs avancés |
| [Why humans are AI's biggest bottleneck](https://www.lennysnewsletter.com/p/why-humans-are-ais-biggest-bottleneck) | Embiricos @ Lenny | Product + Dev |

---
*Notebook créé à partir de [codex-cli-best-practice](https://github.com/shanraisshan/codex-cli-best-practice) — Mai 2026*